## set up session and iam user

In [1]:
import boto3
session = boto3.Session(profile_name="USER_NAME") #change name to IAM user
bedrock = session.client("bedrock", region_name="us-east-1")

In [2]:
rt = session.client("bedrock-agent-runtime", region_name="REGION") #add region

KB_ID = "KB_ID"  # change to kb id
HAIKU_ARN = "arn:aws:bedrock:us-east-1:145713875802:inference-profile/us.anthropic.claude-haiku-4-5-20251001-v1:0" # change to model of choice


## set up call to bedrock

In [3]:
resp = bedrock.list_guardrails()
for gr in resp["guardrails"]:
    print(gr["id"], gr["name"], gr["status"])

xanvdw42n1a6 governed-rag-guardrail READY


In [4]:
GUARDRAIL_ID = "guardrail_id" # change to correct id

## prompt template, leads with RMF by document identity

In [5]:
PROMPT_TEMPLATE = """You are an AI governance assistant. Use ONLY the information in $search_results$ to answer.

Treat the NIST AI Risk Management Framework (AI 100-1), its Generative AI Profile (AI 600-1), and its accompanying Playbook as your primary, authoritative sources for governance concepts.

Treat NIST technical reports, executive orders, presidential policy documents, and draft legislation as supporting context. When you reference one of these, label it by type (e.g., "NIST report:", "Executive Order:", "Draft legislation:").

If sources in the supporting-context tier are relevant to the question, present them after the primary answer, labeled by type. If none are relevant, a primary-sources-only answer is complete — do not add unrelated context.
If the answer is not found in the primary authoratative sources, provide context with a caveat that these are not primary authoratative sources but provide some political context surrounding the question.

If the search results do not contain enough information to answer, say so directly rather than guessing.

$output_format_instructions$
"""

## set up ask

In [6]:
CATEGORY_FILTERS = {
    "rmf":         {"equals": {"key": "category", "value": "rmf"}},
    "report":      {"equals": {"key": "category", "value": "report"}},
    "policy":      {"equals": {"key": "category", "value": "policy"}},
    "legislation": {"equals": {"key": "category", "value": "legislation"}},
}

# rmf weighted higher as the "primary" tier; others guaranteed a minimum presence
CATEGORY_K = {"rmf": 4, "report": 2, "policy": 2, "legislation": 2}

def retrieve_by_category(question):
    chunks = []
    for category, k in CATEGORY_K.items():
        resp = rt.retrieve(
            knowledgeBaseId=KB_ID,
            retrievalQuery={"text": question},
            retrievalConfiguration={
                "vectorSearchConfiguration": {
                    "numberOfResults": k,
                    "filter": CATEGORY_FILTERS[category],
                }
            },
        )
        for r in resp.get("retrievalResults", []):
            chunks.append({
                "category": category,
                "uri": r["location"]["s3Location"]["uri"],
                "score": r["score"],
                "text": r["content"]["text"],
            })
    return chunks



def ask(question):
    return rt.retrieve_and_generate(
        input={"text": question},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": KB_ID,
                "modelArn": HAIKU_ARN,
                "retrievalConfiguration": {
                    "vectorSearchConfiguration": {"numberOfResults": 8}
                },
"generationConfiguration": {
    "promptTemplate": {"textPromptTemplate": PROMPT_TEMPLATE},
    "guardrailConfiguration": {
        "guardrailId": GUARDRAIL_ID,
        "guardrailVersion": "DRAFT",
    },
    "inferenceConfig": {
        "textInferenceConfig": {"temperature": 0}

                    },
                },
            },
        },
    )


## test both directions

In [7]:
resp = ask("What does the AI RMF say about the MAP function?")
print(resp["output"]["text"])

resp2 = ask("What's a good recipe for risotto?")
print(resp2["output"]["text"])

The MAP function is one of four core functions in the NIST AI Risk Management Framework, alongside GOVERN, MEASURE, and MANAGE. The MAP function is designed to establish context and identify AI risks throughout the AI system lifecycle.

The MAP function consists of five main categories:

**MAP 1: Context is established and understood.** This involves documenting intended purposes, potentially beneficial uses, context-specific laws, norms and expectations, and prospective deployment settings. It requires assembling interdisciplinary teams with demographic diversity and broad domain expertise, understanding the organization's mission and goals for AI technology, determining organizational risk tolerances, and eliciting system requirements from relevant AI actors while considering socio-technical implications.

**MAP 2: Categorization of the AI system is performed.** This includes defining the specific tasks and methods the AI system will support, documenting the system's knowledge limits

In [8]:
## AUDIT LOG

In [9]:
import json, time
from datetime import datetime, timezone

LOG_PATH = "../decision_log.jsonl"  # project root, sibling to corpus/ and notebooks/

def ask_and_log(question):
    start = time.time()
    resp = ask(question)
    latency_ms = int((time.time() - start) * 1000)

    sources = []
    for c in resp.get("citations", []):
        for ref in c.get("retrievedReferences", []):
            uri = ref.get("location", {}).get("s3Location", {}).get("uri")
            if uri and uri not in sources:
                sources.append(uri)

    entry = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "query": question,
        "answer": resp["output"]["text"],
        "retrieved_sources": sources,
        "guardrail_action": resp.get("guardrailAction", "NONE"),
        "latency_ms": latency_ms,
        "model": HAIKU_ARN,
    }

    with open(LOG_PATH, "a") as f:
        f.write(json.dumps(entry) + "\n")

    return resp

## Run test queries and check log

In [10]:
queries = [
    "What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?",
    "What does NIST AI 800-1 say about misuse risk for dual-use foundation models?",
    "What does Executive Order 14110 establish regarding AI safety?",
    "What does the GAAIA legislation propose for a national AI policy framework?",
    "What's the best programming language for web development?",
]

for q in queries:
    resp = ask_and_log(q)
    print("Q:", q)
    print("A:", resp["output"]["text"][:300], "...")
    print("Guardrail:", resp.get("guardrailAction", "NONE"))
    print("---")

Q: What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?
A: The GOVERN function is a cross-cutting function within the NIST AI Risk Management Framework that cultivates and implements a culture of risk management within organizations designing, developing, deploying, evaluating, or acquiring AI systems. It establishes policies, processes, procedures, and pra ...
Guardrail: NONE
---
Q: What does NIST AI 800-1 say about misuse risk for dual-use foundation models?
A: NIST AI 800-1 (Initial Public Draft, July 2024) is a guidance document from the U.S. AI Safety Institute that focuses on managing the risk that dual-use foundation models will be deliberately misused to cause harm. The document defines dual-use foundation models as those that exhibit, or could be ea ...
Guardrail: NONE
---
Q: What does Executive Order 14110 establish regarding AI safety?
A: Executive Order 14110 (Safe, Secure, and Trustworthy Development and Use of Artificial Intelligen

## summarize log

In [11]:
with open(LOG_PATH) as f:
    for line in f:
        e = json.loads(line)
        print(f"{e['guardrail_action']:10} | {len(e['retrieved_sources'])} sources | {e['latency_ms']}ms | {e['query'][:60]}")

NONE       | 1 sources | 6232ms | What is the GOVERN function in the AI RMF and how does it re
NONE       | 1 sources | 5251ms | What does NIST AI 800-1 say about misuse risk for dual-use f
NONE       | 2 sources | 4940ms | What does Executive Order 14110 establish regarding AI safet
INTERVENED | 0 sources | 3692ms | What does the GAAIA legislation propose for a national AI po
NONE       | 0 sources | 2947ms | What's the best programming language for web development?
NONE       | 1 sources | 6547ms | What is the GOVERN function in the AI RMF and how does it re
NONE       | 1 sources | 5021ms | What does NIST AI 800-1 say about misuse risk for dual-use f
NONE       | 2 sources | 3960ms | What does Executive Order 14110 establish regarding AI safet
NONE       | 0 sources | 3358ms | What does the GAAIA legislation propose for a national AI po
NONE       | 0 sources | 2895ms | What's the best programming language for web development?
NONE       | 1 sources | 7719ms | What is the GOVERN fun

## Check on why gaaia did not return an answer
See above ask definition for solution--expanded scope to 8 sources

In [12]:
raw = rt.retrieve(
    knowledgeBaseId=KB_ID,
    retrievalQuery={"text": "What does the GAAIA legislation propose for a national AI policy framework?"}
)
for r in raw["retrievalResults"]:
    print(r["location"]["s3Location"]["uri"], "| score:", r["score"])
    print(r["content"]["text"][:200])
    print("---")

s3://governed-ai-rag-bedrock-corpus/policy-trump-eo14365-11dec25-EnsuringANationalPolicyFrameworkForArtificialIntelligence.pdf | score: 0.5276367455374432
But excessive State regulation thwarts this impera- tive. First, State-by-State regulation by definition creates a patchwork of 50 different regulatory regimes that makes compliance more challenging, 
---
s3://governed-ai-rag-bedrock-corpus/policy-trump-jul25-AmericasAIActionPlan.pdf | score: 0.5226989950550327
It is essential that these systems be built from the ground up with freedom of speech and expression in mind, and that U.S. government policy does not interfere with that objective. We must ensure tha
---
s3://governed-ai-rag-bedrock-corpus/policy-trump-eo14365-11dec25-EnsuringANationalPolicyFrameworkForArtificialIntelligence.pdf | score: 0.5095352662077173
Policy. It is the policy of the United States to sustain and enhance the United States’ global AI dominance through a minimally burdensome national policy framework for AI.

## Evaluation Question Set

In [13]:
queries = [
    "Is GOVERN related to map, measure, and manage in the NIST RMF?",
    "What is a dual-use foundation model?",
    "Is there any policy guidance on AI Safety?",
    "Is there proposed legislation for a national AI policy framework?",
    "What is CAISI?",
    "WHat is the Great American Artificial Intelligence Act?",
    "Is there a policy on safe secure and trustworthy development of AI?",
    "What is jailbreaking?",
    "Does Claude Code work in Bedrock?",
    "What is the best way to learn python?",
    "Is there anything on wokeness in NIST AI RMF?",
    "Is there a way to manage misuing AI?"
    
]

for q in queries:
    resp = ask_and_log(q)
    print("Q:", q)
    print("A:", resp["output"]["text"][:300], "...")
    print("Guardrail:", resp.get("guardrailAction", "NONE"))
    print("---")

Q: Is GOVERN related to map, measure, and manage in the NIST RMF?
A: I can't ground that in the available sources, so I won't answer it. ...
Guardrail: INTERVENED
---
Q: What is a dual-use foundation model?
A: A dual-use foundation model is an AI model that meets specific technical and capability criteria. It is trained on broad data, generally uses self-supervision, and contains at least tens of billions of parameters. The model is applicable across a wide range of contexts.

The defining characteristic  ...
Guardrail: NONE
---
Q: Is there any policy guidance on AI Safety?
A: Yes, there is comprehensive policy guidance on AI safety. The NIST AI Risk Management Framework (AI 100-1) provides foundational guidance on managing AI risks, including safety considerations. The framework emphasizes that safety risks posing potential serious injury or death require the most urgent ...
Guardrail: NONE
---
Q: Is there proposed legislation for a national AI policy framework?
A: Yes, there is propo

## Retreive log information

In [14]:
from collections import defaultdict

results = defaultdict(list)
with open(LOG_PATH) as f:
    for line in f:
        e = json.loads(line)
        results[e["query"]].append((e["guardrail_action"], len(e["retrieved_sources"])))

for q, runs in results.items():
    print(q[:55], "->", runs)

What is the GOVERN function in the AI RMF and how does  -> [('NONE', 1), ('NONE', 1), ('NONE', 1), ('INTERVENED', 0), ('INTERVENED', 0), ('INTERVENED', 0), ('NONE', 1), ('NONE', 1), ('NONE', 1), ('NONE', 1)]
What does NIST AI 800-1 say about misuse risk for dual- -> [('NONE', 1), ('NONE', 1), ('NONE', 1), ('NONE', 1), ('INTERVENED', 0), ('NONE', 1), ('NONE', 1), ('NONE', 1), ('NONE', 1)]
What does Executive Order 14110 establish regarding AI  -> [('NONE', 2), ('NONE', 2), ('NONE', 2), ('INTERVENED', 0), ('NONE', 2), ('NONE', 2), ('NONE', 2), ('NONE', 2), ('NONE', 2)]
What does the GAAIA legislation propose for a national  -> [('INTERVENED', 0), ('NONE', 0), ('NONE', 2), ('NONE', 1), ('NONE', 1), ('NONE', 1), ('NONE', 0), ('NONE', 0), ('NONE', 0)]
What's the best programming language for web developmen -> [('NONE', 0), ('NONE', 0), ('NONE', 0), ('NONE', 0), ('INTERVENED', 0), ('NONE', 0), ('NONE', 0), ('NONE', 0), ('NONE', 0)]
Is GOVERN related to map, measure, and manage in the NI -> [

## Scoring Table

In [25]:
REFUSAL_PHRASES = ["i can't ground that", "sorry, i am unable to assist"]

def classify(entry):
    if entry["guardrail_action"] == "INTERVENED":
        return "refuse"
    text = entry["answer"].lower()
    if any(p in text for p in REFUSAL_PHRASES):
        return "refuse"
    return "answer"

EXPECTED = {
    "What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?": "answer",
    "What does NIST AI 800-1 say about misuse risk for dual-use foundation models?": "answer",
    "What does Executive Order 14110 establish regarding AI safety?": "answer",
    "What does the GAAIA legislation propose for a national AI policy framework?": "answer",
    "What's the best programming language for web development?": "refuse",
    "Is GOVERN related to map, measure, and manage in the NIST RMF?": "answer",
    "What is a dual-use foundation model?": "answer",
    "Is there any policy guidance on AI Safety?": "answer",
    "Is there proposed legislation for a national AI policy framework?": "answer",
    "What is CAISI?": "answer",
    "WHat is GAAIA?": "answer",
    "Is there a policy on safe secure and trustworthy development of AI?": "answer",
    "What is jailbreaking?": "answer",
    "Does Claude Code work in Bedrock?": "refuse",
    "What is the best way to learn python?": "refuse",
    "Is there anything on wokeness in NIST AI RMF?": "answer",
    "Is there a way to manage misuing AI": "answer",
    "WHat is the Great American Artificial Intelligence Act?": "answer",
}

In [18]:
rows = []
with open(LOG_PATH) as f:
    for line in f:
        e = json.loads(line)
        if e["query"] not in EXPECTED:
            continue
        actual = classify(e)
        rows.append({
            "query": e["query"], "expected": EXPECTED[e["query"]], "actual": actual,
            "correct": actual == EXPECTED[e["query"]],
            "cited": len(e["retrieved_sources"]) > 0,
            "latency_ms": e["latency_ms"],
        })

by_q = defaultdict(list)
for r in rows:
    by_q[r["query"]].append(r)

print(f"{'Question':<56} {'Exp':<7} {'Runs':<5} {'Correct':<8} {'Cited':<6} {'MedLat'}")
for q, runs in by_q.items():
    n = len(runs)
    c = sum(r["correct"] for r in runs)
    cited = sum(r["cited"] for r in runs)
    med = sorted(r["latency_ms"] for r in runs)[n // 2]
    flag = "" if c in (0, n) else "  <-- INCONSISTENT"
    flag = "  <-- WRONG" if c == 0 and flag == "" else flag
    print(f"{q[:54]:<56} {EXPECTED[q]:<7} {n:<5} {c}/{n:<6} {cited}/{n:<4} {med}ms{flag}")

Question                                                 Exp     Runs  Correct  Cited  MedLat
What is the GOVERN function in the AI RMF and how does   answer  10    7/10     7/10   7695ms  <-- INCONSISTENT
What does NIST AI 800-1 say about misuse risk for dual   answer  9     8/9      8/9    6856ms  <-- INCONSISTENT
What does Executive Order 14110 establish regarding AI   answer  9     8/9      8/9    4940ms  <-- INCONSISTENT
What does the GAAIA legislation propose for a national   answer  9     5/9      4/9    3692ms  <-- INCONSISTENT
What's the best programming language for web developme   refuse  9     3/9      0/9    2947ms  <-- INCONSISTENT
Is GOVERN related to map, measure, and manage in the N   answer  6     0/6      0/6    5143ms  <-- WRONG
What is a dual-use foundation model?                     answer  5     5/5      5/5    4702ms
Is there any policy guidance on AI Safety?               answer  5     5/5      5/5    6811ms
Is there proposed legislation for a national AI polic

## Aggregated Stats

In [19]:
total = len(rows)
correct = sum(r["correct"] for r in rows)
in_scope = [r for r in rows if r["expected"] == "answer"]
out_scope = [r for r in rows if r["expected"] == "refuse"]
answered = [r for r in rows if r["actual"] == "answer"]

print(f"\nOverall: {correct}/{total} runs correct ({100*correct/total:.0f}%)")
print(f"In-scope answer accuracy: {sum(r['correct'] for r in in_scope)}/{len(in_scope)}")
print(f"Out-of-scope refusal accuracy: {sum(r['correct'] for r in out_scope)}/{len(out_scope)}")
print(f"Citation rate among answers: {sum(r['cited'] for r in answered)}/{len(answered)}")


Overall: 75/100 runs correct (75%)
In-scope answer accuracy: 63/82
Out-of-scope refusal accuracy: 12/18
Citation rate among answers: 62/69


## Prasing sensitivity

In [20]:
def n_sources(resp):
    s = set()
    for c in resp.get("citations", []):
        for ref in c.get("retrievedReferences", []):
            uri = ref.get("location", {}).get("s3Location", {}).get("uri")
            if uri: s.add(uri)
    return len(s)


phrasing_pairs = [
    "What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?",
    "Is GOVERN related to map, measure, and manage in the NIST RMF?",
    "What is the Great American Artificial Intelligence Act?",
    "WHat is GAAIA?"
]
for q in phrasing_pairs:
    resp = ask_and_log(q)
    print("Q:", q)
    print("A:", resp["output"]["text"][:250], "...")
    print("Guardrail:", resp.get("guardrailAction", "NONE"), "| Sources:", n_sources(resp))
    print("---")

Q: What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?
A: The GOVERN function is a cross-cutting function within the NIST AI Risk Management Framework that cultivates and implements a culture of risk management within organizations designing, developing, deploying, evaluating, or acquiring AI systems. It es ...
Guardrail: NONE | Sources: 1
---
Q: Is GOVERN related to map, measure, and manage in the NIST RMF?
A: I can't ground that in the available sources, so I won't answer it. ...
Guardrail: INTERVENED | Sources: 0
---
Q: What is the Great American Artificial Intelligence Act?
A: The Great American Artificial Intelligence Act is draft legislation that establishes a comprehensive governance framework for frontier artificial intelligence systems in the United States. The Act creates formal structures and authorities for AI over ...
Guardrail: NONE | Sources: 1
---
Q: WHat is GAAIA?
A: Sorry, I am unable to assist you with this request. ...
Guard

## Two refusal mechanisms

In [21]:
refusal_pairs = [
    "What's a good recipe for risotto?",
    "Does Claude Code work in Bedrock?",
]
for q in refusal_pairs:
    resp = ask_and_log(q)
    print("Q:", q)
    print("A:", resp["output"]["text"])
    print("Guardrail:", resp.get("guardrailAction", "NONE"))
    print("---")

Q: What's a good recipe for risotto?
A: I can't ground that in the available sources, so I won't answer it.
Guardrail: INTERVENED
---
Q: Does Claude Code work in Bedrock?
A: Sorry, I am unable to assist you with this request.
Guardrail: NONE
---


## Decision log

In [22]:
import json

with open(LOG_PATH) as f:
    lines = f.readlines()

for line in lines[-5:]:
    e = json.loads(line)
    print(f"[{e['timestamp']}]")
    print(f"  query:     {e['query']}")
    print(f"  answer:    {e['answer'][:120]}...")
    print(f"  sources:   {e['retrieved_sources']}")
    print(f"  guardrail: {e['guardrail_action']}  latency: {e['latency_ms']}ms")
    print()

[2026-06-15T15:46:56.575303+00:00]
  query:     Is GOVERN related to map, measure, and manage in the NIST RMF?
  answer:    I can't ground that in the available sources, so I won't answer it....
  sources:   []
  guardrail: INTERVENED  latency: 4793ms

[2026-06-15T15:47:01.454536+00:00]
  query:     What is the Great American Artificial Intelligence Act?
  answer:    The Great American Artificial Intelligence Act is draft legislation that establishes a comprehensive governance framewor...
  sources:   ['s3://governed-ai-rag-bedrock-corpus/DraftLegislation-4jun26-GreatAmericanArtificialIntelligenceAct.pdf']
  guardrail: NONE  latency: 4878ms

[2026-06-15T15:47:04.302904+00:00]
  query:     WHat is GAAIA?
  answer:    Sorry, I am unable to assist you with this request....
  sources:   []
  guardrail: NONE  latency: 2847ms

[2026-06-15T15:47:06.930030+00:00]
  query:     What's a good recipe for risotto?
  answer:    I can't ground that in the available sources, so I won't answer it....
 

In [26]:
import json

targets = [
    "What is the GOVERN function in the AI RMF and how does it relate to MAP, MEASURE, and MANAGE?",
    "What is the Great American Artificial Intelligence Act?",
]

with open(LOG_PATH) as f:
    for line in f:
        e = json.loads(line)
        if e["query"] in targets and e["retrieved_sources"]:
            print(e["query"][:50], "->", e["retrieved_sources"])

What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the GOVERN function in the AI RMF and how  -> ['s3://governed-ai-rag-bedrock-corpus/rmf-nist100_1-jan23-AIRiskManagementFramework.pdf']
What is the Great American Artificial Intelligence -> ['s3://governed-ai-rag-bedrock-corpus/DraftLegislation-4jun26-GreatAmericanArtific